# Filtro de Datos 

Con el objetivo de reducir la cantidad de informacion a graficar, decidimos hacer un filtro dejando solo los 2 mejores y los 2 peores y uno extra personal. Este ultimo, elegido este ultimo elegido por ser potencia o tener relacion con el usuario



In [7]:
import pandas as pd

# Cargar json
df = pd.read_json("datos_limpios.json")

# Diccionario país -> continente
continentes = {
    # América
    "Argentina": "América",
    "Bolivia": "América",
    "Brasil": "América",
    "Chile": "América",
    "Colombia": "América",
    "Costa Rica": "América",
    "Cuba": "América",
    "Ecuador": "América",
    "El Salvador": "América",
    "Guatemala": "América",
    "Haití": "América",
    "Honduras": "América",
    "México": "América",
    "Nicaragua": "América",
    "Panamá": "América",
    "Paraguay": "América",
    "Perú": "América",
    "República Dominicana": "América",
    "Uruguay": "América",
    "Venezuela": "América",
    "Estados Unidos": "América",
    "Canadá": "América",

    # Europa
    "Alemania": "Europa",
    "Reino Unido": "Europa",
    "Francia": "Europa",
    "Italia": "Europa",
    "España": "Europa",
    "Suecia": "Europa",
    "Noruega": "Europa",
    "Suiza": "Europa",
    "Grecia": "Europa",
    "Polonia": "Europa",
    "Ucrania": "Europa",
    "Rusia": "Europa",

    # Asia
    "China": "Asia",
    "India": "Asia",
    "Japón": "Asia",
    "Corea del Sur": "Asia",
    "Arabia Saudita": "Asia",
    "Turquía": "Asia",
    "Indonesia": "Asia",
    "Tailandia": "Asia",
    "Vietnam": "Asia",
    "Pakistán": "Asia",
    "Filipinas": "Asia",
    "Hong Kong": "Asia",

    # África
    "Sudáfrica": "África",
    "Egipto": "África",
    "Marruecos": "África",
    "Nigeria": "África",

    # Oceanía
    "Australia": "Oceanía",
    "Nueva Zelanda": "Oceanía"
}
# Agregar columna continente
df["Continente"] = df["País"].map(continentes)

In [8]:
df.head()

,País,Sedán Más Vendido (Combustión),Consumo Mixto Aprox.,Precio Aprox. Litro Gasolina (USD),Litros cargados con 50 dolares,Kilometros cargados con 50 dolares,iso_alpha,Continente
0,Argentina,Fiat Cronos,13.5,1.520,32.894737,444.078947,ARG,América
1,Bolivia,Suzuki Dzire,19.5,1.008,49.603175,967.261905,BOL,América
2,Brasil,Chevrolet Onix Plus,16.5,1.352,36.982249,610.207101,NaN,América
3,Chile,Nissan Versa,15.0,1.706,29.308324,439.624853,CHL,América
4,Colombia,Renault Logan,13.0,1.137,43.975374,571.679859,COL,América


In [12]:
col = "Kilometros cargados con 50 dolares"

mejores = (
    df.groupby("Continente")
      .apply(lambda x: x.nlargest(2, col))
      .reset_index(level=0)
)

peores = (
    df.groupby("Continente")
      .apply(lambda x: x.nsmallest(2, col))
      .reset_index(level=0)
)

resultado = pd.concat([mejores, peores], ignore_index=True)

resultado = resultado.sort_values(
    ["Continente", col],
    ascending=[True, False]
)

print(resultado[["Continente", "País", col]])

   Continente           País  Kilometros cargados con 50 dolares
0     América        Bolivia                          967.261905
1     América       Paraguay                          720.260223
11    América          Chile                          439.624853
10    América        Uruguay                          403.620352
2        Asia          India                         1036.866359
3        Asia          Japón                          687.855787
13       Asia  Corea del Sur                          351.170569
12       Asia      Hong Kong                          169.286578
4      Europa        Polonia                          485.865724
5      Europa        Ucrania                          476.327945
15     Europa         Grecia                          298.108553
14     Europa       Alemania                          293.284790
6     Oceanía      Australia                          501.036628
17    Oceanía      Australia                          501.036628
7     Oceanía  Nueva Zela

In [13]:
resultado.head()

,Continente,País,Sedán Más Vendido (Combustión),Consumo Mixto Aprox.,Precio Aprox. Litro Gasolina (USD),Litros cargados con 50 dolares,Kilometros cargados con 50 dolares,iso_alpha
0,América,Bolivia,Suzuki Dzire,19.5,1.008,49.603175,967.261905,BOL
1,América,Paraguay,Kia Soluto,15.5,1.076,46.468401,720.260223,PRY
11,América,Chile,Nissan Versa,15.0,1.706,29.308324,439.624853,CHL
10,América,Uruguay,Chevrolet Onix Plus,16.5,2.044,24.461840,403.620352,URY
2,Asia,India,Maruti Suzuki Dzire,22.5,1.085,46.082949,1036.866359,IND


In [14]:
# agregar países por decision propia (nueva zelanda, estados unidos, china, rusia, hong kong)
resultado = pd.concat([resultado, df[df["País"].isin(["Nueva Zelanda", "Estados Unidos", "China", "Rusia", "Hong Kong"])]])

#  sacar posibles duplicados en caso de que un país esté tanto en top 2 como en bottom 2 (aunque no debería pasar)

resultado = resultado.drop_duplicates(subset=["País"])


In [15]:
# pasar este dataframe a archivo json
resultado.to_json("datos_a_utilizar.json", orient="records", force_ascii=False, indent=2)